In [ ]:
import importlib
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "agentic_rag.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "agentic_rag.py").exists():
    raise FileNotFoundError("Open this notebook inside the agentic-rag-v2 project.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

import agentic_rag

importlib.reload(agentic_rag)
answer_with_agent = agentic_rag.answer_with_agent
conversation = []


def ask_book(question: str):
    answer, sources, activity = answer_with_agent(question, history=conversation)
    conversation.extend(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    )

    print("Agent tool activity:")
    for step in activity:
        action_input = step.get("input", step.get("query", "(no input)"))
        result = step.get("result", f"{step.get('matches', 0)} matches")
        print(f"- {step.get('tool', 'tool')}: {action_input} -> {result}")

    print("\nAnswer:\n", answer)
    if sources:
        print("\nSources:")
        for source in sources:
            print(
                f"- [{source['id']}] {source['section']} "
                f"(pages {source['page_start']}-{source['page_end']})"
            )
    else:
        print("No book sources were used.")


ask_book("What is mind management?")